In [ ]:

# CALCULADORA DE MUESTRAS ESTADÍSTICAS

import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math, Markdown, HTML, clear_output


CUSTOM_CSS = """
<style>
    .calc-container {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        max-width: 900px;
        margin: 0 auto;
    }
    .calc-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 25px 30px;
        border-radius: 15px;
        margin-bottom: 20px;
        text-align: center;
        box-shadow: 0 10px 30px rgba(102, 126, 234, 0.3);
    }
    .calc-header h1 { margin: 0; font-size: 28px; font-weight: 700; }
    .calc-header p { margin: 5px 0 0; opacity: 0.9; font-size: 14px; }
    .section-card {
        background: #f8f9ff;
        border: 1px solid #e2e6f3;
        border-radius: 12px;
        padding: 20px 25px;
        margin-bottom: 15px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.04);
    }
    .section-card h3 {
        color: #4a5568;
        margin-top: 0;
        font-size: 16px;
        border-bottom: 2px solid #667eea;
        padding-bottom: 8px;
        display: inline-block;
    }
    .param-table {
        width: 100%;
        border-collapse: collapse;
        font-size: 14px;
        margin-top: 10px;
    }
    .param-table th {
        background: #667eea;
        color: white;
        padding: 10px 15px;
        text-align: left;
        font-weight: 600;
    }
    .param-table td {
        padding: 9px 15px;
        border-bottom: 1px solid #e8ecf4;
    }
    .param-table tr:nth-child(even) { background: #eef0fa; }
    .param-table tr:hover { background: #dde1f5; }
    .result-box {
        background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%);
        color: #1a3a2a;
        padding: 20px 25px;
        border-radius: 12px;
        margin-top: 15px;
        text-align: center;
        box-shadow: 0 8px 25px rgba(67, 233, 123, 0.25);
    }
    .result-box h2 { margin: 0 0 5px; font-size: 42px; font-weight: 800; }
    .result-box p { margin: 0; font-size: 15px; font-weight: 500; }
    .result-detail {
        background: #fff;
        border-radius: 10px;
        padding: 15px 20px;
        margin-top: 10px;
        font-size: 13px;
        text-align: left;
        color: #4a5568;
    }
    .result-detail code {
        background: #eef0fa;
        padding: 2px 6px;
        border-radius: 4px;
        font-size: 13px;
    }
    .warning-box {
        background: #fff3cd;
        border: 1px solid #ffc107;
        border-radius: 10px;
        padding: 12px 18px;
        margin-top: 10px;
        font-size: 13px;
        color: #856404;
    }
    .info-box {
        background: #e8f4fd;
        border: 1px solid #bee5eb;
        border-radius: 10px;
        padding: 12px 18px;
        margin-top: 10px;
        font-size: 13px;
        color: #0c5460;
    }
</style>
"""

# Valores Z para niveles de confianza comunes
Z_VALUES = {
    "90%": 1.645,
    "95%": 1.96,
    "97%": 2.17,
    "99%": 2.576,
    "99.9%": 3.291,
}

SAMPLING_TYPES = {
    # ------------------------------------------------------------------
    # 1. MUESTREO ALEATORIO SIMPLE
    # ------------------------------------------------------------------
    "Muestreo Aleatorio Simple": {
        "description": (
            "Cada elemento de la población tiene la misma probabilidad de ser "
            "seleccionado. Es el método más básico y sirve de referencia para "
            "los demás diseños muestrales."
        ),
        "formula_finite": r"n = \frac{Z^2 \cdot p \cdot q \cdot N}{e^2 \cdot (N-1) + Z^2 \cdot p \cdot q}",
        "formula_infinite": r"n_0 = \frac{Z^2 \cdot p \cdot q}{e^2}",
        "params": [
            ("N", "Tamaño de la población", "Número total de elementos en la población de estudio"),
            ("Z", "Valor Z (nivel de confianza)", "Valor de la distribución normal estándar asociado al nivel de confianza deseado"),
            ("p", "Proporción esperada de éxito", "Proporción estimada del atributo en la población (usar 0.5 si se desconoce para máxima variabilidad)"),
            ("q", "Proporción complementaria (1 − p)", "Complemento de p. Se calcula automáticamente como q = 1 − p"),
            ("e", "Margen de error (precisión)", "Error máximo aceptable en la estimación, expresado como proporción (ej: 0.05 = 5%)"),
        ],
    },
    # ------------------------------------------------------------------
    # 2. MUESTREO ESTRATIFICADO
    # ------------------------------------------------------------------
    "Muestreo Estratificado": {
        "description": (
            "La población se divide en subgrupos homogéneos (estratos) y se "
            "selecciona una muestra de cada estrato. Esto mejora la precisión "
            "y asegura la representación de cada subgrupo."
        ),
        "formula_prop": r"n = \frac{N \cdot \sum_{h=1}^{L} W_h \cdot p_h \cdot q_h}{\frac{N \cdot e^2}{Z^2} + \sum_{h=1}^{L} W_h \cdot p_h \cdot q_h}",
        "formula_alloc": r"n_h = n \cdot W_h = n \cdot \frac{N_h}{N}",
        "params": [
            ("N", "Tamaño total de la población", "Suma de todos los elementos en todos los estratos"),
            ("L", "Número de estratos", "Cantidad de subgrupos en que se divide la población"),
            ("N_h", "Tamaño del estrato h", "Número de elementos en cada estrato"),
            ("W_h", "Peso del estrato (N_h / N)", "Proporción que representa cada estrato respecto al total"),
            ("p_h", "Proporción esperada en el estrato h", "Proporción del atributo dentro de cada estrato"),
            ("Z", "Valor Z (nivel de confianza)", "Valor de la distribución normal para el nivel de confianza"),
            ("e", "Margen de error deseado", "Error máximo aceptable para la estimación global"),
        ],
    },
    # ------------------------------------------------------------------
    # 3. MUESTREO SISTEMÁTICO
    # ------------------------------------------------------------------
    "Muestreo Sistemático": {
        "description": (
            "Se selecciona un elemento al azar como punto de partida y luego "
            "se elige cada k-ésimo elemento de la lista. Es sencillo de "
            "implementar y proporciona una cobertura uniforme de la población."
        ),
        "formula_k": r"k = \frac{N}{n}",
        "formula_n": r"n = \frac{Z^2 \cdot p \cdot q \cdot N}{e^2 \cdot (N-1) + Z^2 \cdot p \cdot q}",
        "params": [
            ("N", "Tamaño de la población", "Número total de elementos en el marco muestral"),
            ("n", "Tamaño de muestra", "Se calcula con la fórmula de muestreo aleatorio simple"),
            ("k", "Intervalo de selección", "Se selecciona un elemento cada k posiciones; k = N/n"),
            ("Z", "Valor Z (nivel de confianza)", "Valor de la distribución normal estándar"),
            ("p", "Proporción esperada de éxito", "Proporción estimada del atributo en la población"),
            ("e", "Margen de error", "Error máximo aceptable en la estimación"),
        ],
    },
    # ------------------------------------------------------------------
    # 4. MUESTREO POR CONGLOMERADOS
    # ------------------------------------------------------------------
    "Muestreo por Conglomerados": {
        "description": (
            "La población se divide en grupos naturales (conglomerados), como "
            "escuelas, barrios o empresas. Se seleccionan aleatoriamente "
            "algunos conglomerados y se estudian todos sus elementos. Se "
            "aplica un factor de corrección llamado Efecto de Diseño (DEFF)."
        ),
        "formula_base": r"n_0 = \frac{Z^2 \cdot p \cdot q}{e^2}",
        "formula_deff": r"n_{ajustado} = n_0 \cdot DEFF",
        "formula_final": r"n = \frac{n_{ajustado}}{1 + \frac{n_{ajustado} - 1}{N}}",
        "params": [
            ("N", "Tamaño de la población", "Número total de elementos en la población"),
            ("Z", "Valor Z (nivel de confianza)", "Valor de la distribución normal estándar"),
            ("p", "Proporción esperada de éxito", "Proporción estimada del atributo en la población"),
            ("e", "Margen de error", "Error máximo aceptable"),
            ("DEFF", "Efecto de diseño", "Factor de corrección que mide la pérdida de eficiencia al usar conglomerados vs. MAS. Generalmente DEFF ≥ 1 (típicamente entre 1.5 y 3)"),
        ],
    },
    # ------------------------------------------------------------------
    # 5. MUESTREO PARA PROPORCIONES
    # ------------------------------------------------------------------
    "Muestreo para Proporciones": {
        "description": (
            "Se utiliza cuando el objetivo es estimar la proporción de una "
            "característica en la población (ej: % de personas que aprueban "
            "una política). Incluye corrección por finitud cuando la población "
            "es conocida."
        ),
        "formula_infinite": r"n_0 = \frac{Z^2 \cdot p \cdot (1-p)}{e^2}",
        "formula_finite": r"n = \frac{n_0}{1 + \frac{n_0 - 1}{N}}",
        "params": [
            ("N", "Tamaño de la población (si es finita)", "Número total de elementos. Dejar en 0 si la población es infinita o muy grande"),
            ("Z", "Valor Z (nivel de confianza)", "Valor de la distribución normal estándar"),
            ("p", "Proporción esperada", "Proporción estimada del atributo. Usar 0.5 para la máxima variabilidad cuando no se tiene información previa"),
            ("e", "Margen de error", "Error máximo aceptable, expresado como proporción"),
        ],
    },
}

# ============================================================================
# FUNCIONES DE CÁLCULO
# ============================================================================

def calc_aleatorio_simple(N, Z, p, e):
    """Muestreo Aleatorio Simple — población finita e infinita."""
    q = 1 - p
    n0 = (Z**2 * p * q) / (e**2)  # fórmula para población infinita
    if N > 0:
        n = (Z**2 * p * q * N) / (e**2 * (N - 1) + Z**2 * p * q)
    else:
        n = n0
    return {
        "n0": math.ceil(n0),
        "n": math.ceil(n),
        "q": q,
        "finite": N > 0,
    }


def calc_estratificado(N, Z, e, strata):
    """
    Muestreo Estratificado con afijación proporcional.
    strata: lista de tuplas (N_h, p_h) para cada estrato.
    """
    L = len(strata)
    total_Nh = sum(s[0] for s in strata)
    if total_Nh != N:
        N = total_Nh  # ajustar al total real de estratos

    sum_wpq = 0
    strata_info = []
    for Nh, ph in strata:
        Wh = Nh / N
        qh = 1 - ph
        sum_wpq += Wh * ph * qh
        strata_info.append({
            "Nh": Nh, "Wh": round(Wh, 4),
            "ph": ph, "qh": round(qh, 4),
        })

    numerator = N * sum_wpq
    denominator = (N * e**2 / Z**2) + sum_wpq
    n_total = math.ceil(numerator / denominator)

    # Afijación proporcional por estrato
    for s in strata_info:
        s["nh"] = math.ceil(n_total * s["Wh"])

    return {
        "n": n_total,
        "L": L,
        "strata": strata_info,
        "sum_wpq": round(sum_wpq, 6),
    }


def calc_sistematico(N, Z, p, e):
    """Muestreo Sistemático: calcula n (como MAS) y el intervalo k."""
    result = calc_aleatorio_simple(N, Z, p, e)
    n = result["n"]
    k = N / n if n > 0 else 0
    return {
        "n": n,
        "k": round(k, 2),
        "k_int": math.floor(k),
        "q": result["q"],
    }


def calc_conglomerados(N, Z, p, e, deff):
    """Muestreo por Conglomerados con efecto de diseño."""
    q = 1 - p
    n0 = (Z**2 * p * q) / (e**2)
    n_adj = n0 * deff
    if N > 0:
        n = n_adj / (1 + (n_adj - 1) / N)
    else:
        n = n_adj
    return {
        "n0": math.ceil(n0),
        "n_adj": math.ceil(n_adj),
        "n": math.ceil(n),
        "q": q,
        "deff": deff,
        "finite": N > 0,
    }


def calc_proporciones(N, Z, p, e):
    """Muestreo para Proporciones con corrección de finitud."""
    n0 = (Z**2 * p * (1 - p)) / (e**2)
    if N > 0:
        n = n0 / (1 + (n0 - 1) / N)
    else:
        n = n0
    return {
        "n0": math.ceil(n0),
        "n": math.ceil(n),
        "finite": N > 0,
    }


# ============================================================================
# GRÁFICA DE SENSIBILIDAD
# ============================================================================

def plot_sensitivity(Z, p, N, calc_type="simple", deff=1.5):
    """Genera gráfica de cómo varía n con el margen de error."""
    errors = np.arange(0.01, 0.11, 0.005)
    sizes = []

    for err in errors:
        if calc_type == "conglomerados":
            res = calc_conglomerados(N, Z, p, err, deff)
        elif calc_type == "proporciones":
            res = calc_proporciones(N, Z, p, err)
        else:
            res = calc_aleatorio_simple(N, Z, p, err)
        sizes.append(res["n"])

    fig, ax = plt.subplots(figsize=(9, 4.5))

    # Estilo premium
    ax.set_facecolor("#f8f9ff")
    fig.patch.set_facecolor("#ffffff")

    # Línea principal con gradiente simulado
    ax.fill_between(errors * 100, sizes, alpha=0.15, color="#667eea")
    ax.plot(errors * 100, sizes, color="#667eea", linewidth=2.5,
            marker="o", markersize=4, markerfacecolor="#764ba2",
            markeredgecolor="white", markeredgewidth=1.5)

    # Línea de referencia para e=5%
    idx_5 = list(errors).index(0.05) if 0.05 in errors else None
    if idx_5 is not None:
        ax.axvline(x=5, color="#e74c3c", linestyle="--", alpha=0.5, linewidth=1)
        ax.annotate(f"e=5% → n={sizes[idx_5]}",
                    xy=(5, sizes[idx_5]),
                    xytext=(6.5, sizes[idx_5] * 1.1),
                    fontsize=10, color="#e74c3c", fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.5))

    ax.set_xlabel("Margen de Error (%)", fontsize=12, fontweight="600", color="#4a5568")
    ax.set_ylabel("Tamaño de Muestra (n)", fontsize=12, fontweight="600", color="#4a5568")
    ax.set_title("📈 Análisis de Sensibilidad: Tamaño de Muestra vs. Margen de Error",
                 fontsize=13, fontweight="700", color="#2d3748", pad=15)

    ax.grid(True, alpha=0.3, linestyle="--")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#cbd5e0")
    ax.spines["bottom"].set_color("#cbd5e0")

    plt.tight_layout()
    plt.show()



def create_ui():
    """Crea y despliega la interfaz interactiva con ipywidgets."""

    # --- Mostrar encabezado ---
    display(HTML(CUSTOM_CSS))
    display(HTML("""
    <div class="calc-container">
        <div class="calc-header">
            <h1>📊 Calculadora de Muestras Estadísticas</h1>
            <p>Seleccione un tipo de muestreo, configure los parámetros y obtenga el tamaño de muestra óptimo</p>
        </div>
    </div>
    """))

    # --- Widgets ---
    sampling_dropdown = widgets.Dropdown(
        options=list(SAMPLING_TYPES.keys()),
        value="Muestreo Aleatorio Simple",
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
    )

    confidence_dropdown = widgets.Dropdown(
        options=list(Z_VALUES.keys()),
        value="95%",
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
    )

    N_input = widgets.IntText(
        value=10000, description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
    )

    p_input = widgets.FloatSlider(
        value=0.5, min=0.01, max=0.99, step=0.01,
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
        readout_format=".2f",
    )

    e_input = widgets.FloatSlider(
        value=0.05, min=0.01, max=0.10, step=0.005,
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
        readout_format=".3f",
    )

    deff_input = widgets.FloatSlider(
        value=1.5, min=1.0, max=5.0, step=0.1,
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
        readout_format=".1f",
    )

    # Estratos (para muestreo estratificado)
    num_strata_input = widgets.IntSlider(
        value=3, min=2, max=10, step=1,
        description="",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "0px"},
    )

    calculate_btn = widgets.Button(
        description="🔢  Calcular Tamaño de Muestra",
        button_style="primary",
        layout=widgets.Layout(width="100%", height="45px"),
        style={"font_weight": "bold"},
    )

    output_area = widgets.Output()
    formula_area = widgets.Output()
    params_area = widgets.Output()
    inputs_area = widgets.Output()

    # Contenedores para estratos dinámicos
    strata_inputs_container = widgets.VBox()
    strata_inputs = []

    def build_strata_inputs(num):
        nonlocal strata_inputs
        strata_inputs = []
        children = []
        for i in range(num):
            nh = widgets.IntText(value=1000, layout=widgets.Layout(width="45%"))
            ph = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01,
                                     layout=widgets.Layout(width="45%"), readout_format=".2f")
            row = widgets.HBox([
                widgets.HTML(f"<b style='min-width:80px;color:#4a5568'>Estrato {i+1}:</b>"),
                widgets.VBox([widgets.HTML("<small>N<sub>h</sub></small>"), nh],
                             layout=widgets.Layout(width="45%")),
                widgets.VBox([widgets.HTML("<small>p<sub>h</sub></small>"), ph],
                             layout=widgets.Layout(width="45%")),
            ])
            strata_inputs.append((nh, ph))
            children.append(row)
        strata_inputs_container.children = children

    build_strata_inputs(3)

    def on_num_strata_change(change):
        build_strata_inputs(change["new"])
    num_strata_input.observe(on_num_strata_change, names="value")

    # --- Mostrar fórmula y parámetros ---
    def show_formula_and_params(sampling_type):
        st = SAMPLING_TYPES[sampling_type]

        with formula_area:
            clear_output(wait=True)
            display(HTML(f"""
            <div class="section-card">
                <h3>📖 Descripción</h3>
                <p style="color:#4a5568; font-size:14px; line-height:1.6">{st['description']}</p>
            </div>
            """))

            display(HTML('<div class="section-card"><h3>📐 Fórmula(s)</h3>'))
            # Mostrar todas las fórmulas disponibles
            formula_keys = [k for k in st.keys() if k.startswith("formula")]
            for fk in formula_keys:
                label = fk.replace("formula_", "").replace("_", " ").title()
                display(HTML(f"<p style='color:#667eea;font-weight:600;margin:8px 0 2px;font-size:13px'>▸ {label}:</p>"))
                display(Math(st[fk]))
            display(HTML("</div>"))

        with params_area:
            clear_output(wait=True)
            rows = ""
            for sym, name, desc in st["params"]:
                rows += f"<tr><td><b style='color:#667eea;font-family:serif;font-size:16px'>{sym}</b></td><td><b>{name}</b></td><td>{desc}</td></tr>"

            display(HTML(f"""
            <div class="section-card">
                <h3>📋 Parámetros</h3>
                <table class="param-table">
                    <tr><th>Símbolo</th><th>Nombre</th><th>Descripción</th></tr>
                    {rows}
                </table>
            </div>
            """))

    # --- Actualizar campos de entrada según tipo ---
    def update_inputs(sampling_type):
        with inputs_area:
            clear_output(wait=True)
            is_strat = sampling_type == "Muestreo Estratificado"
            is_cong = sampling_type == "Muestreo por Conglomerados"

            common_inputs = widgets.VBox([
                widgets.HTML("<b style='color:#4a5568'>🔒 Nivel de Confianza:</b>"),
                confidence_dropdown,
                widgets.HTML("<b style='color:#4a5568'>👥 Tamaño de la Población (N):</b>"),
                widgets.HTML("<small style='color:#718096'>Ingrese 0 si la población es infinita o muy grande</small>"),
                N_input,
            ])

            if is_strat:
                specific = widgets.VBox([
                    widgets.HTML("<b style='color:#4a5568'>📊 Margen de Error (e):</b>"),
                    e_input,
                    widgets.HTML("<b style='color:#4a5568'>🔢 Número de Estratos:</b>"),
                    num_strata_input,
                    widgets.HTML("<b style='color:#4a5568'>📋 Configuración de Estratos:</b>"),
                    strata_inputs_container,
                ])
            elif is_cong:
                specific = widgets.VBox([
                    widgets.HTML("<b style='color:#4a5568'>📊 Proporción esperada (p):</b>"),
                    p_input,
                    widgets.HTML("<b style='color:#4a5568'>📏 Margen de Error (e):</b>"),
                    e_input,
                    widgets.HTML("<b style='color:#4a5568'>🔧 Efecto de Diseño (DEFF):</b>"),
                    widgets.HTML("<small style='color:#718096'>Valores típicos: 1.5 a 3.0. Mayor DEFF = mayor tamaño de muestra</small>"),
                    deff_input,
                ])
            else:
                specific = widgets.VBox([
                    widgets.HTML("<b style='color:#4a5568'>📊 Proporción esperada (p):</b>"),
                    widgets.HTML("<small style='color:#718096'>Use 0.50 si no tiene datos previos (máxima variabilidad)</small>"),
                    p_input,
                    widgets.HTML("<b style='color:#4a5568'>📏 Margen de Error (e):</b>"),
                    e_input,
                ])

            display(HTML('<div class="section-card"><h3>⚙️ Configurar Parámetros</h3>'))
            display(common_inputs)
            display(specific)
            display(HTML('</div>'))
            display(calculate_btn)

    # --- Calcular ---
    def on_calculate(btn):
        with output_area:
            clear_output(wait=True)
            sampling_type = sampling_dropdown.value
            Z = Z_VALUES[confidence_dropdown.value]
            N = N_input.value
            p = p_input.value
            e = e_input.value

            try:
                if sampling_type == "Muestreo Aleatorio Simple":
                    res = calc_aleatorio_simple(N, Z, p, e)
                    n_result = res["n"]
                    detail = f"""
                    <b>Datos utilizados:</b><br>
                    <code>N = {N:,}</code> | <code>Z = {Z}</code> ({confidence_dropdown.value}) |
                    <code>p = {p}</code> | <code>q = {res['q']}</code> | <code>e = {e}</code><br><br>
                    <b>Muestra sin corrección (población infinita):</b> n₀ = {res['n0']:,}<br>
                    <b>Muestra con corrección de finitud:</b> n = {res['n']:,}
                    """
                    calc_t = "simple"

                elif sampling_type == "Muestreo Estratificado":
                    strata = [(nh.value, ph.value) for nh, ph in strata_inputs]
                    res = calc_estratificado(N, Z, e, strata)
                    n_result = res["n"]
                    strata_detail = ""
                    for i, s in enumerate(res["strata"]):
                        strata_detail += (
                            f"Estrato {i+1}: N<sub>h</sub>={s['Nh']:,}, "
                            f"W<sub>h</sub>={s['Wh']}, p<sub>h</sub>={s['ph']}, "
                            f"<b>n<sub>h</sub>={s['nh']}</b><br>"
                        )
                    detail = f"""
                    <b>Datos utilizados:</b><br>
                    <code>N = {N:,}</code> | <code>Z = {Z}</code> ({confidence_dropdown.value}) |
                    <code>e = {e}</code> | <code>Estratos = {res['L']}</code><br><br>
                    <b>Distribución por estratos (afijación proporcional):</b><br>
                    {strata_detail}
                    """
                    calc_t = "simple"

                elif sampling_type == "Muestreo Sistemático":
                    res = calc_sistematico(N, Z, p, e)
                    n_result = res["n"]
                    detail = f"""
                    <b>Datos utilizados:</b><br>
                    <code>N = {N:,}</code> | <code>Z = {Z}</code> ({confidence_dropdown.value}) |
                    <code>p = {p}</code> | <code>e = {e}</code><br><br>
                    <b>Tamaño de muestra:</b> n = {res['n']:,}<br>
                    <b>Intervalo de selección:</b> k = {res['k']} ≈ {res['k_int']}<br>
                    <b>Instrucción:</b> Seleccione un inicio aleatorio entre 1 y {res['k_int']},
                    luego seleccione cada {res['k_int']}-ésimo elemento.
                    """
                    calc_t = "simple"

                elif sampling_type == "Muestreo por Conglomerados":
                    deff = deff_input.value
                    res = calc_conglomerados(N, Z, p, e, deff)
                    n_result = res["n"]
                    detail = f"""
                    <b>Datos utilizados:</b><br>
                    <code>N = {N:,}</code> | <code>Z = {Z}</code> ({confidence_dropdown.value}) |
                    <code>p = {p}</code> | <code>e = {e}</code> | <code>DEFF = {deff}</code><br><br>
                    <b>Muestra base (MAS):</b> n₀ = {res['n0']:,}<br>
                    <b>Ajustado por DEFF:</b> n × DEFF = {res['n_adj']:,}<br>
                    <b>Con corrección de finitud:</b> n = {res['n']:,}
                    """
                    calc_t = "conglomerados"

                elif sampling_type == "Muestreo para Proporciones":
                    res = calc_proporciones(N, Z, p, e)
                    n_result = res["n"]
                    detail = f"""
                    <b>Datos utilizados:</b><br>
                    <code>N = {N:,}</code> | <code>Z = {Z}</code> ({confidence_dropdown.value}) |
                    <code>p = {p}</code> | <code>e = {e}</code><br><br>
                    <b>Muestra sin corrección:</b> n₀ = {res['n0']:,}<br>
                    <b>Con corrección de finitud:</b> n = {res['n']:,}
                    """
                    calc_t = "proporciones"

                # Mostrar resultado
                display(HTML(f"""
                <div class="result-box">
                    <p>TAMAÑO DE MUESTRA REQUERIDO</p>
                    <h2>n = {n_result:,}</h2>
                    <p>{sampling_type} — Confianza: {confidence_dropdown.value}</p>
                    <div class="result-detail">{detail}</div>
                </div>
                """))

                # Validaciones y advertencias
                if N > 0 and n_result > N * 0.5:
                    display(HTML("""
                    <div class="warning-box">
                        ⚠️ <b>Advertencia:</b> El tamaño de muestra supera el 50% de la población.
                        Considere realizar un censo o aumentar el margen de error.
                    </div>
                    """))

                if N > 0 and n_result > N:
                    display(HTML("""
                    <div class="warning-box">
                        🚫 <b>Atención:</b> El tamaño de muestra calculado excede la población.
                        Debe encuestar a toda la población (censo).
                    </div>
                    """))

                display(HTML("""
                <div class="info-box">
                    💡 <b>Nota:</b> El tamaño de muestra se redondeó al entero superior
                    para garantizar la precisión deseada.
                </div>
                """))

                # Gráfica de sensibilidad
                display(HTML("<br>"))
                plot_sensitivity(Z, p, N, calc_type=calc_t,
                                 deff=deff_input.value if calc_t == "conglomerados" else 1.5)

            except Exception as ex:
                display(HTML(f"""
                <div class="warning-box">
                    ❌ <b>Error en el cálculo:</b> {str(ex)}<br>
                    Verifique que los parámetros ingresados sean válidos.
                </div>
                """))

    calculate_btn.on_click(on_calculate)

    # --- Reaccionar al cambio de tipo de muestreo ---
    def on_type_change(change):
        with output_area:
            clear_output()
        show_formula_and_params(change["new"])
        update_inputs(change["new"])

    sampling_dropdown.observe(on_type_change, names="value")

    # --- Layout principal ---
    display(HTML('<div class="section-card"><h3>🎯 Tipo de Muestreo</h3>'))
    display(sampling_dropdown)
    display(HTML('</div>'))

    display(formula_area)
    display(params_area)
    display(inputs_area)
    display(output_area)

    # Mostrar estado inicial
    show_formula_and_params("Muestreo Aleatorio Simple")
    update_inputs("Muestreo Aleatorio Simple")



if __name__ == "__main__":
    create_ui()


Dropdown(layout=Layout(width='100%'), options=('Muestreo Aleatorio Simple', 'Muestreo Estratificado', 'Muestre…

Output()

Output()

Output()

Output()